# AVG v2 Feature Extraction — CIFAR-10

Load the saved AVG v2 CNN and extract aligned train, validation, and official test embeddings. This notebook does not fit models or score test accuracy.


## 1. Import dan Reproducibility

`SEED = 30092026` harus sama dengan RGB dan AVG Training supaya sample alignment tetap konsisten.

In [ ]:
import os
import random
import numpy as np

SEED = 30092026

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf

# Lebih aman untuk GPU dengan VRAM terbatas.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

tf.random.set_seed(SEED)

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Model, load_model
from sklearn.model_selection import train_test_split

print("SEED:", SEED)
print("GPU :", gpus)

## 2. Konfigurasi File

In [ ]:
CNN_MODEL_PATH = "cifar10_avg_cnn_v2.keras"
FEATURE_OUTPUT_PATH = "avg_features_v2.npz"
FEATURE_LAYER = "svm_features"

if not os.path.exists(CNN_MODEL_PATH):
    raise FileNotFoundError(
        f"{CNN_MODEL_PATH} tidak ditemukan. "
        "Jalankan AVG Training terlebih dahulu dan letakkan file model "
        "pada folder yang sama dengan notebook ini."
    )

print("CNN model     :", CNN_MODEL_PATH)
print("Feature output:", FEATURE_OUTPUT_PATH)
print("Feature layer :", FEATURE_LAYER)

## 3. Load CIFAR-10, Grayscale AVG, dan Recreate Split

Split dibuat dengan `SEED = 30092026` dan stratification yang sama.

Setelah split ditentukan, RGB images dikonversi menjadi:

```text
AVG = (R + G + B) / 3
```

`train_indices` dan `val_indices` juga disimpan di archive agar alignment dengan RGB dan NTSC dapat diverifikasi.

In [ ]:
(X_train_full_rgb, y_train_full), (X_test_rgb, y_test) = cifar10.load_data()

y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

# Simpan exact sample indices untuk future feature fusion.
all_indices = np.arange(len(X_train_full_rgb))

train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full
)

# Ambil sample menggunakan exact indices yang sama.
X_train_rgb = X_train_full_rgb[train_indices]
X_val_rgb = X_train_full_rgb[val_indices]

y_train = y_train_full[train_indices]
y_val = y_train_full[val_indices]

# RGB -> Grayscale AVG -> normalisasi.
X_train = np.mean(
    X_train_rgb.astype("float32"),
    axis=-1,
    keepdims=True
) / 255.0

X_val = np.mean(
    X_val_rgb.astype("float32"),
    axis=-1,
    keepdims=True
) / 255.0

X_test = np.mean(
    X_test_rgb.astype("float32"),
    axis=-1,
    keepdims=True
) / 255.0

print("AVG Training   :", X_train.shape, y_train.shape)
print("AVG Validation :", X_val.shape, y_val.shape)
print("AVG Test       :", X_test.shape, y_test.shape)

assert X_train.shape == (40000, 32, 32, 1)
assert X_val.shape == (10000, 32, 32, 1)
assert X_test.shape == (10000, 32, 32, 1)

## 4. Load Trained Grayscale AVG CNN

`compile=False` digunakan karena model hanya dipakai untuk inference / feature extraction.

In [ ]:
tf.keras.backend.clear_session()

cnn_model = load_model(
    CNN_MODEL_PATH,
    compile=False
)

if FEATURE_LAYER not in [layer.name for layer in cnn_model.layers]:
    raise ValueError(
        f"Layer '{FEATURE_LAYER}' tidak ditemukan pada CNN."
    )

print("AVG CNN berhasil dimuat.")
print("Input shape        :", cnn_model.input_shape)
print(
    "Feature layer shape:",
    cnn_model.get_layer(FEATURE_LAYER).output.shape
)

## 5. Buat AVG CNN Feature Extractor

Output diambil dari layer `svm_features`.

```text
RGB image
    ↓
Grayscale AVG
    ↓
AVG CNN
    ↓
svm_features
    ↓
512-dimensional AVG feature vector
```

In [ ]:
feature_extractor = Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer(
        FEATURE_LAYER
    ).output,
    name="avg_cnn_feature_extractor"
)

feature_extractor.summary()

## 6. Extract AVG Features

Feature extraction dilakukan untuk:
- training set,
- validation set,
- official test set.

Hasilnya menjadi matrix 512 feature per image.

> Jika VRAM penuh, shutdown kernel notebook training/testing lain sebelum menjalankan extraction.

In [ ]:
BATCH_SIZE = 128

X_train_features = feature_extractor.predict(
    X_train,
    batch_size=BATCH_SIZE,
    verbose=1
)

X_val_features = feature_extractor.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=1
)

X_test_features = feature_extractor.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

X_train_features = X_train_features.astype(np.float32)
X_val_features = X_val_features.astype(np.float32)
X_test_features = X_test_features.astype(np.float32)

print()
print("AVG train features :", X_train_features.shape)
print("AVG val features   :", X_val_features.shape)
print("AVG test features  :", X_test_features.shape)

## 7. Validasi AVG Feature Matrix

Layer `svm_features` menghasilkan 512-dimensional feature vector untuk setiap image.

In [ ]:
FEATURE_DIM = int(
    cnn_model.get_layer(FEATURE_LAYER).output.shape[-1]
)

assert X_train_features.shape == (
    len(X_train),
    FEATURE_DIM
)

assert X_val_features.shape == (
    len(X_val),
    FEATURE_DIM
)

assert X_test_features.shape == (
    len(X_test),
    FEATURE_DIM
)

assert len(y_train) == len(X_train_features)
assert len(y_val) == len(X_val_features)
assert len(y_test) == len(X_test_features)

assert np.isfinite(X_train_features).all()
assert np.isfinite(X_val_features).all()
assert np.isfinite(X_test_features).all()

print("Feature dimension :", FEATURE_DIM)
print("Semua shape valid.")
print("Tidak ditemukan NaN/Inf pada extracted features.")

## 8. Simpan ke `avg_features_v2.npz`

Archive menyimpan:
- `X_train_features`
- `X_val_features`
- `X_test_features`
- `y_train`
- `y_val`
- `y_test`
- `train_indices`
- `val_indices`
- metadata (`seed`, `feature_dim`, `feature_layer`, `representation`, `grayscale_formula`)

Indices dan labels memungkinkan kita memastikan RGB, AVG, dan NTSC benar-benar merepresentasikan sample yang sama sebelum feature concatenation.

In [ ]:
np.savez_compressed(
    FEATURE_OUTPUT_PATH,
    X_train_features=X_train_features,
    X_val_features=X_val_features,
    X_test_features=X_test_features,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    train_indices=train_indices,
    val_indices=val_indices,
    seed=np.array(SEED, dtype=np.int64),
    feature_dim=np.array(FEATURE_DIM, dtype=np.int64),
    feature_layer=np.array(FEATURE_LAYER),
    representation=np.array("AVG"),
    grayscale_formula=np.array("(R + G + B) / 3")
)

print("AVG feature archive berhasil disimpan:")
print(FEATURE_OUTPUT_PATH)

## 9. Verifikasi File Hasil

In [ ]:
with np.load(FEATURE_OUTPUT_PATH) as avg:
    print("Keys:")
    for key in avg.files:
        print(
            f"- {key}:",
            avg[key].shape
            if avg[key].ndim > 0
            else avg[key].item()
        )

    assert np.array_equal(
        avg["train_indices"],
        train_indices
    )
    assert np.array_equal(
        avg["val_indices"],
        val_indices
    )
    assert np.array_equal(
        avg["y_train"],
        y_train
    )
    assert np.array_equal(
        avg["y_val"],
        y_val
    )
    assert np.array_equal(
        avg["y_test"],
        y_test
    )

print()
print("Verifikasi selesai: avg_features_v2.npz valid.")

# Selesai — AVG Feature Extraction

Output:

```text
avg_features_v2.npz
```

Artifact Grayscale AVG:

```text
cifar10_avg_cnn_v2.keras
cifar10_avg_svm_v2.joblib
avg_features_v2.npz
```

Pada tugas Feature Engineering nanti:

```text
rgb_features_v2.npz
        +
avg_features_v2.npz
        +
ntsc_features.npz
        ↓
RGB + AVG + NTSC feature fusion
```